# 10M Transformer: 3-Digit Addition in JAX + Flax + Optax)


In [ ]:
!pip install -q flax optax


In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
from flax.training import train_state
import time
import random

print('JAX devices:', jax.devices())
print('JAX version:', jax.__version__)


Vocabulary & Tokenisation

In [ ]:
CHARS      = list('0123456789 +=')
VOCAB_SIZE = len(CHARS)          # 13


stoi = {c:i for i, c in enumerate(CHARS)}
itos = {i:c for i, c in enumerate(CHARS)}
PAD_TOK = "<PAD>"
itos[VOCAB_SIZE] = PAD_TOK
stoi[PAD_TOK] = VOCAB_SIZE
PAD_ID = VOCAB_SIZE

# 999 + 999 = 1998 (16 chars) + 4 to be able to make experiments with bigger numbers
SEQ_LEN = 20

# Methods
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[idx] for idx in ids if idx != VOCAB_SIZE)

# Sanity checks
for ch in CHARS: 
    i = stoi[ch]
    print(f"{ch} maps to {i} and {i} maps to {itos[i]}")

s = "100 + 200 = 300"
print('Sample encode/decode:', decode(encode(s)))

0 maps to 0 and 0 maps to 0
1 maps to 1 and 1 maps to 1
2 maps to 2 and 2 maps to 2
3 maps to 3 and 3 maps to 3
4 maps to 4 and 4 maps to 4
5 maps to 5 and 5 maps to 5
6 maps to 6 and 6 maps to 6
7 maps to 7 and 7 maps to 7
8 maps to 8 and 8 maps to 8
9 maps to 9 and 9 maps to 9
  maps to 10 and 10 maps to  
+ maps to 11 and 11 maps to +
= maps to 12 and 12 maps to =
Sample encode/decode: 100 + 200 = 300


Dataset


It will simply contain all string from "1 + 1 = 2" until "999 + 999 = 1899"

[0, 1, 2]

In [ ]:
import random
import numpy as np

X = []
for i in range(1000):
    for j in range(1000):
        X.append(f"{i} + {j} = {i+j}")


def make_example(x):
    
    # tokenize
    tokens = encode(x)

    # Pad until SEQ_LEN
    padded = tokens + [PAD_ID] * (SEQ_LEN+1 - len(tokens))
    input = np.array(padded[:-1])
    target = np.array(padded[1:])

    # Mask
    equality_idx = x.index("=")
    mask = np.zeros(SEQ_LEN)
    mask[equality_idx+1:] = 1
    
    return input, target, mask



# sanity check
inp, trg, msk = make_example(x_train[1])
print(x_train[1], len(inp), len(trg), len(msk))

279 + 700 = 979 17 17 17


In [ ]:
# Build the Dataset
random.shuffle(X)
examples = [make_example(ex) for ex in X]
Input, Target, Mask = [ex for ex in zip(*examples)]

split = int(len(X)*0.85)
inp_train = Input[:split]
trg_train = Target[:split]
msk_train = Mask[:split]
inp_val = Input[split:]
trg_val = Target[split:]
msk_val = Mask[split:]


print(f'Train: {len(inp_train)}   Val: {len(inp_val)}')


Train: 850,000   Val: 150,000


Model: 10M transformer-decoder

Optimizer

Training-Loop

Inference

Benchmarking